# LLM-Augmented Black-Litterman Views — An Out-of-Sample Comparative Study

**Abstract.** This notebook evaluates five Black-Litterman view-generator variants over a fully out-of-sample window (2023-01 → 2026-04) on a 29-asset cross-asset universe (equities, fixed income, commodities, real assets). All variants share identical BL infrastructure — τ = 0.05, δ = 2.5, Ledoit-Wolf covariance estimator, monthly refits, long-only weights — and differ only in how the view vector **Q** is formed:

| Generator | View source |
|---|---|
| `equilibrium` | No views — BL collapses to CAPM-equilibrium MSR |
| `momentum` | Annualised 252-day trailing return per asset |
| `mean_rev` | −1 × annualised 5-year mean (sign-flip reversion) |
| `llm-anthropic` | LLM (`claude-opus-4-7`) given trailing return evidence |
| `llm-openai` | LLM (`gpt-5.5`) given identical trailing return evidence |

**Governance principle.** The LLM emits view vectors (Q, confidence); Black-Litterman is the allocator. The LLM never sets portfolio weights directly.

**Headline caveat.** Both LLM training cutoffs fall inside the backtest window (Anthropic ≈ 2026-01, OpenAI ≈ mid-2025). Section 7 tests whether apparent LLM edge reflects genuine point-in-time skill or training-data hindsight.

**To reproduce:**
```bash
python scripts/run_bl_views_backtest.py --generator equilibrium
python scripts/run_bl_views_backtest.py --generator momentum
python scripts/run_bl_views_backtest.py --generator mean_rev
python scripts/run_bl_views_backtest.py --generator llm-anthropic
python scripts/run_bl_views_backtest.py --generator llm-openai
```

In [ ]:
import sys
sys.path.insert(0, "../src")

import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

from aiam.evaluation.performance import performance_stats

ROOT = Path("..").resolve()
RESULTS_DIR = ROOT / "results" / "llm"

GENERATORS = ["equilibrium", "momentum", "mean_rev", "llm-anthropic", "llm-openai"]
DISPLAY = {
    "equilibrium":  "BL-Equilibrium",
    "momentum":     "BL-Momentum",
    "mean_rev":     "BL-MeanRev",
    "llm-anthropic": "BL-LLM(Anthropic)",
    "llm-openai":   "BL-LLM(OpenAI)",
}
COLORS = {
    "equilibrium":  "#1f77b4",
    "momentum":     "#ff7f0e",
    "mean_rev":     "#2ca02c",
    "llm-anthropic": "#d62728",
    "llm-openai":   "#9467bd",
}

plt.rcParams.update({"figure.dpi": 120, "font.size": 10})
print(f"Results directory: {RESULTS_DIR}")

## 1 — How the LLM forms a view — a worked example

For one representative refit date — **2024-02-01** — this section traces the full
evidence → prompt → views → (P, Q, Ω) pipeline for both LLM providers.  
The same cached API responses used in the main backtest are replayed here;
**no new API calls are made**.


In [ ]:
# Step 1 — Evidence packet the LLM sees
DEMO_DATE = pd.Timestamp("2024-02-01")

_prices_demo = pd.read_parquet(ROOT / "data/cache/prices_29.parquet")
_prices_demo = _prices_demo[_prices_demo.index.dayofweek < 5]
_returns_demo = _prices_demo.pct_change().dropna(how="all")
_hist_demo = _returns_demo.loc[_returns_demo.index <= DEMO_DATE]

from aiam.llm.evidence import build_evidence, evidence_to_text

_evidence_df = build_evidence(_hist_demo, DEMO_DATE)
print(f"Evidence packet  date={DEMO_DATE.date()}  "
      f"({len(_evidence_df)} assets × {len(_evidence_df.columns)} features)\n")
display(
    _evidence_df.style
    .format("{:+.4f}")
    .background_gradient(cmap="RdYlGn", subset=["ret_252d"], vmin=-0.3, vmax=0.3)
    .set_caption("Trailing returns (21d / 63d / 252d) and annualised volatility "
                 "passed to the LLM — no look-ahead")
)


In [ ]:
# Step 2 — System + user prompt (excerpt)
from aiam.llm.prompts import SYSTEM_PROMPT, build_prompt

_evidence_text = evidence_to_text(_evidence_df)
_user_prompt   = build_prompt(_evidence_text, DEMO_DATE, list(_returns_demo.columns))

print("── SYSTEM PROMPT ────────────────────────────────────────────────────────")
print(SYSTEM_PROMPT[:420])
print("  [...] (full text in src/aiam/llm/prompts.py)")
print()
print("── USER PROMPT (first 580 chars) ────────────────────────────────────────")
print(_user_prompt[:580])
print("  [...] (table continues for all 29 assets)")


In [ ]:
# Step 3 — Replay from cache; parse ViewSets for both providers
from aiam.llm.cache import PromptCache
from aiam.llm.schemas import parse_viewset

_demo_cache = PromptCache(ROOT / "data/cache/llm")
_providers  = [("llm-anthropic", "claude-opus-4-7"), ("llm-openai", "gpt-5.5")]

_viewsets = {}
for gen, model in _providers:
    raw = _demo_cache.get(model, SYSTEM_PROMPT, _user_prompt)
    assert raw is not None, f"Cache miss for {gen} on {DEMO_DATE.date()}"
    _viewsets[gen] = parse_viewset(raw)

# Side-by-side comparison table
_an_map = {v.asset: v for v in _viewsets["llm-anthropic"].views}
_oi_map = {v.asset: v for v in _viewsets["llm-openai"].views}
_all_assets = sorted(set(_an_map) | set(_oi_map))

_rows_vs = []
for a in _all_assets:
    va, vo = _an_map.get(a), _oi_map.get(a)
    _rows_vs.append({
        "Asset":     a,
        "Anth EER":  va.expected_excess_return if va else None,
        "Anth conf": va.confidence             if va else None,
        "OAI EER":   vo.expected_excess_return if vo else None,
        "OAI conf":  vo.confidence             if vo else None,
    })

_vs_df = pd.DataFrame(_rows_vs).set_index("Asset")

for gen, vs in _viewsets.items():
    print(f"{DISPLAY[gen]:25s} {len(vs.views):2d} views   rationale: {vs.rationale}")
print()
display(
    _vs_df.style
    .format("{:+.4f}", subset=["Anth EER", "OAI EER"], na_rep="—")
    .format("{:.2f}",  subset=["Anth conf", "OAI conf"], na_rep="—")
    .background_gradient(cmap="RdYlGn",
                         subset=["Anth EER", "OAI EER"], vmin=-0.30, vmax=0.30)
    .set_caption(
        f"ViewSets — {DEMO_DATE.date()} — identical evidence, both providers.  "
        "EER = expected excess return (annualised).  '—' = no view expressed."
    )
)


In [ ]:
# Step 4 — (P, Q, Ω) matrices produced by LLMViewGenerator for each provider
class _DemoCacheOnlyClient:
    def __init__(self, model): self.model = model
    def complete(self, *a, **kw):
        raise RuntimeError("cache-only mode")

from aiam.llm.views import LLMViewGenerator

_pqo = {}
for gen, model in _providers:
    _gen_obj = LLMViewGenerator(_DemoCacheOnlyClient(model), cache=_demo_cache)
    _pqo[gen] = _gen_obj(_hist_demo, DEMO_DATE)

_assets_all = list(_returns_demo.columns)
fig, axes = plt.subplots(1, 2, figsize=(17, 6))
for ax, (gen, (P, Q, Omega)) in zip(axes, _pqo.items()):
    _view_assets = [_assets_all[int(np.argmax(P[i]))] for i in range(len(Q))]
    _df_pqo = (
        pd.DataFrame({"Q (ann. EER)": Q, "Ω diag": np.diag(Omega)},
                     index=_view_assets)
        .sort_values("Q (ann. EER)", ascending=True)
    )
    _color = COLORS.get(gen, "#888")
    _df_pqo["Q (ann. EER)"].plot.barh(ax=ax, color=_color, width=0.65, alpha=0.85)
    ax.axvline(0, color="#555", linewidth=0.8)
    ax.set_title(f"{DISPLAY[gen]}\nView vector Q  ({DEMO_DATE.date()})", fontsize=12)
    ax.set_xlabel("Expected excess return (annualised)", fontsize=11)
    ax.tick_params(axis="both", labelsize=10)
    ax.text(0.02, -0.14, f"P: {P.shape}   Q: {Q.shape}   Ω: {Omega.shape}",
            transform=ax.transAxes, fontsize=9, color="#666")
    ax.grid(True, axis="x", alpha=0.25)
fig.tight_layout()
plt.show()

print()
print(f"{'Provider':28s} {'k views':>8} {'mean|Q|':>10} {'mean Ω_diag':>14}")
print("-" * 64)
for gen, (P, Q, Omega) in _pqo.items():
    print(f"{DISPLAY[gen]:28s} {len(Q):8d} {np.mean(np.abs(Q)):10.4f} "
          f"{np.mean(np.diag(Omega)):14.2e}")

### What the worked example shows

**Identical evidence in.** Both providers receive the same 29-asset packet:
21d, 63d, and 252d trailing returns plus annualised volatility, as-of 2024-02-01.
No market commentary, news, or macro data is supplied — only realised return history.

**Agreement.** Both models tilt toward US large-cap tech momentum (NVDA, MSFT, XLK, SPY)
and fade China equities (FXI) and energy (XLE, XOM) — consistent with the prevailing
momentum factor environment visible in the evidence itself.

**Divergence.** OpenAI covers 21 assets (broader universe, including fixed-income and
commodities); Anthropic selects 13 with higher average confidence on fewer positions.
The NVDA view diverges materially: +0.25 / conf 0.55 (Anthropic) vs. +0.45 / conf 0.78
(OpenAI) — same momentum signal, different magnitude and certainty calibration.

**Governance.** The LLM outputs only (P, Q, Ω): a pick-matrix, a view vector, and a
per-view uncertainty diagonal. Black-Litterman blends these with the equilibrium prior
and the full covariance matrix to form posterior expected returns; a mean-variance
optimizer then sets portfolio weights. **The LLM never allocates capital directly.**


## 2 — Load results

In [ ]:
returns_dict: dict[str, pd.Series] = {}
diag_dict: dict[str, dict] = {}

for gen in GENERATORS:
    rpath = RESULTS_DIR / f"{gen}_returns.parquet"
    dpath = RESULTS_DIR / f"{gen}_diagnostics.json"
    if rpath.exists():
        df = pd.read_parquet(rpath)
        returns_dict[gen] = df["portfolio_return"]
        if dpath.exists():
            diag_dict[gen] = json.loads(dpath.read_text())
        print(f"  {DISPLAY[gen]:25s} {len(returns_dict[gen]):>5d} OOS days  "
              f"mock={diag_dict.get(gen, {}).get('mock', '?')}")
    else:
        print(f"  [MISSING] {rpath.name}")

print(f"\nLoaded {len(returns_dict)}/{len(GENERATORS)} generators.")

## 3 — Performance comparison table

In [ ]:
rows = []
for gen, ret in returns_dict.items():
    diag = diag_dict.get(gen, {})
    s = performance_stats(ret.dropna())
    rows.append({
        "Strategy":       DISPLAY[gen],
        "Sharpe":         s["sharpe_ratio"],
        "Ann Ret":        s["annualized_return"],
        "Ann Vol":        s["annualized_volatility"],
        "Max DD":         s["max_drawdown"],
        "Turnover/refit": diag.get("mean_turnover_per_refit"),
        "Views/refit":    diag.get("mean_views_per_refit"),
        "Refits":         diag.get("n_refits"),
        "Note":           "[mock]" if diag.get("mock") else "",
    })

# ML bar reference
rows.append({
    "Strategy": "── ML bar (session 3 best) ──",
    "Sharpe": 2.579, "Ann Ret": None, "Ann Vol": None,
    "Max DD": None, "Turnover/refit": None, "Views/refit": None,
    "Refits": None, "Note": "reference",
})

# Top strategies from existing ML comparison table for context
ml_csv = ROOT / "data/cache/portfolio_returns/ml_strategies_comparison_table.csv"
if ml_csv.exists():
    ml_df = pd.read_csv(ml_csv)
    for _, r in ml_df.head(3).iterrows():
        rows.append({
            "Strategy": r["Strategy"],
            "Sharpe": r["Sharpe"],
            "Ann Ret": r.get("Ann Ret"),
            "Ann Vol": r.get("Ann Vol"),
            "Max DD":  r.get("Max DD"),
            "Turnover/refit": None, "Views/refit": None,
            "Refits": None, "Note": "prior ML/classical",
        })

comp = (
    pd.DataFrame(rows)
    .sort_values("Sharpe", ascending=False, na_position="last")
    .reset_index(drop=True)
)

fmt = {
    "Sharpe":         "{:.3f}",
    "Ann Ret":        "{:.3f}",
    "Ann Vol":        "{:.3f}",
    "Max DD":         "{:.3f}",
    "Turnover/refit": "{:.4f}",
    "Views/refit":    "{:.1f}",
}
comp.style.format(fmt, na_rep="—").set_caption("BL view-generator comparison (OOS 2023-01-01 →)")

## 4 — Equity curves

In [ ]:
if not returns_dict:
    print("No results to plot. Run the backtest scripts first.")
else:
    fig, ax = plt.subplots(figsize=(15, 7.5))

    for gen, ret in returns_dict.items():
        cum = (1 + ret.dropna()).cumprod()
        label = DISPLAY[gen]
        if diag_dict.get(gen, {}).get("mock"):
            label += " [mock]"
        ax.plot(cum.index, cum.values, label=label, color=COLORS[gen], linewidth=1.6)

    ax.axhline(1.0, color="#888", linewidth=0.8, linestyle="--")
    ax.set_title("BL view-generator variants — OOS equity curves", fontsize=13)
    ax.set_xlabel("Date", fontsize=12)
    ax.set_ylabel("Growth of $1", fontsize=12)
    ax.tick_params(axis="both", labelsize=11)
    ax.legend(loc="upper left", fontsize=11)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    plt.show()

## 5 — Drawdown comparison

In [ ]:
if not returns_dict:
    print("No results to plot.")
else:
    fig, ax = plt.subplots(figsize=(15, 5))

    for gen, ret in returns_dict.items():
        cum = (1 + ret.dropna()).cumprod()
        dd = (cum - cum.cummax()) / cum.cummax()
        ax.fill_between(dd.index, dd.values, 0, alpha=0.25, color=COLORS[gen])
        ax.plot(dd.index, dd.values, color=COLORS[gen], linewidth=0.8, label=DISPLAY[gen])

    ax.set_title("Drawdown", fontsize=13)
    ax.set_xlabel("Date", fontsize=12)
    ax.set_ylabel("Drawdown", fontsize=12)
    ax.tick_params(axis="both", labelsize=11)
    ax.legend(loc="lower left", fontsize=10)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    plt.show()

## 6 — LLM view diagnostics

In [ ]:
llm_gens = [g for g in ["llm-anthropic", "llm-openai"] if g in diag_dict]

if not llm_gens:
    print("No LLM results yet.\n"
          "Run:\n"
          "  python scripts/run_bl_views_backtest.py --generator llm-anthropic\n"
          "  python scripts/run_bl_views_backtest.py --generator llm-openai")
else:
    # Summary table
    llm_rows = []
    for gen in llm_gens:
        d = diag_dict[gen]
        llm_rows.append({
            "Generator":      DISPLAY[gen],
            "Refits":         d.get("n_refits"),
            "Views/refit":    d.get("mean_views_per_refit"),
            "Turnover/refit": d.get("mean_turnover_per_refit"),
            "Sharpe":         d.get("sharpe_ratio"),
            "Mock":           d.get("mock"),
        })
    display(pd.DataFrame(llm_rows))

    # Load per-refit weights to show mean allocation by asset over time
    fig, axes = plt.subplots(1, len(llm_gens), figsize=(8.5 * len(llm_gens), 5.5))
    if len(llm_gens) == 1:
        axes = [axes]

    for ax, gen in zip(axes, llm_gens):
        wpath = RESULTS_DIR / f"{gen}_weights.parquet"
        if not wpath.exists():
            ax.text(0.5, 0.5, "weights missing", ha="center", transform=ax.transAxes)
            continue
        wdf = pd.read_parquet(wpath)
        # Top 8 assets by mean allocation
        top8 = wdf.mean().nlargest(8).index.tolist()
        wdf[top8].plot.bar(ax=ax, stacked=False, width=0.7, legend=True)
        ax.set_title(f"{DISPLAY[gen]}\nmean weight per refit (top 8 assets)", fontsize=11)
        ax.set_xlabel("Refit date", fontsize=10)
        ax.set_ylabel("Weight", fontsize=10)
        ax.tick_params(axis="x", rotation=45, labelsize=9)
        ax.tick_params(axis="y", labelsize=9)
        ax.legend(fontsize=9, loc="upper right")

    fig.tight_layout()
    plt.show()

## 7 — Turnover and refit count summary

In [ ]:
if not diag_dict:
    print("No diagnostics available yet.")
else:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    gens_avail = [g for g in GENERATORS if g in diag_dict]
    labels = [DISPLAY[g] for g in gens_avail]
    sharpes = [diag_dict[g].get("sharpe_ratio") or 0.0 for g in gens_avail]
    turnovers = [diag_dict[g].get("mean_turnover_per_refit") or 0.0 for g in gens_avail]
    bar_colors = [COLORS[g] for g in gens_avail]

    ax1.barh(labels, sharpes, color=bar_colors)
    ax1.axvline(2.579, color="black", linestyle="--", linewidth=1, label="ML bar")
    ax1.set_title("OOS Sharpe ratio", fontsize=12)
    ax1.tick_params(axis="both", labelsize=11)
    ax1.legend(fontsize=10)
    ax1.grid(True, axis="x", alpha=0.3)

    ax2.barh(labels, turnovers, color=bar_colors)
    ax2.set_title("Mean turnover per refit", fontsize=12)
    ax2.tick_params(axis="both", labelsize=11)
    ax2.grid(True, axis="x", alpha=0.3)

    fig.tight_layout()
    plt.show()

## Findings

*OOS 2023-01-01 → 2026-04-30 · 833 trading days · 41 monthly refits · 29-asset universe · BL τ=0.05, δ=2.5, Ledoit-Wolf covariance · returns-only evidence.*

### Summary

| Rank | Generator | Sharpe | Ann Ret | Ann Vol | Max DD | Turn/refit | Views/refit |
|---|---|---|---|---|---|---|---|
| 1 | BL-LLM(Opus 4.7) | **2.284** | 0.357 | 0.138 | −0.097 | 0.445 | 13.0 |
| 2 | BL-Equilibrium | 2.037 | 0.222 | 0.101 | −0.122 | 0.000 | 0 |
| 3 | BL-LLM(GPT-5.5) | 1.979 | 0.271 | 0.125 | −0.110 | 0.508 | 15.8 |
| 4 | BL-Momentum | 1.869 | 0.327 | 0.158 | −0.156 | 0.212 | 29 |
| 5 | BL-MeanRev | 1.560 | 0.278 | 0.166 | −0.123 | 0.476 | 29 |
| ref | ML bar (session 3 best) | 2.579 | — | — | — | — | — |

ML/classical top-3 for context: VMP(MDP(LW)) 2.422 · MSR(RF_μ̂) 2.394 · SignalTilt(XGB) 2.304 (Ann Ret 0.707 / Ann Vol 0.245 / MaxDD −0.226 — much rougher drawdown than BL-LLM(Opus)).

### Key questions — answers

**1. Do LLM views add Sharpe versus the rule-based BL variants?**
Yes, for both providers over both rule-based generators. Opus beats momentum by +0.416 and mean_rev by +0.725 Sharpe. Both LLMs also outperform equilibrium, though GPT-5.5 does so only marginally (+0.058). View quality is model-dependent: Opus adds a clear margin; GPT-5.5 barely clears the no-views prior.

**2. Does BL-LLM clear or approach the 2.579 ML bar?**
Not cleared. Opus reaches 2.284 (gap −0.295), placing it within the ML/classical cluster range (VMP 2.422, MSR(RF) 2.394). BL-LLM(Opus) matches SignalTilt(XGB) on Sharpe (≈2.3) with substantially better drawdown (−0.097 vs −0.226). That smoothness is a BL property, not a lookahead artefact.

**3. Which LLM provider produces more consistent/higher-quality views?**
Anthropic (claude-opus-4-7) is clearly superior: Sharpe 2.284 vs 1.979, fewer but higher-quality views (13 vs 16 per refit), better drawdown (−0.097 vs −0.110), and higher IC (+0.117 vs +0.105). Neither provider shows a hindsight fingerprint; Opus's edge appears to reflect better confidence calibration and selectivity.

**4. Is turnover reasonable relative to transaction cost budget (~10 bps)?**
Moderate concern. LLM generators average 0.445–0.508 one-way turnover per monthly refit, implying ≈60–70 bps round-trip/month — 6–7× the 10 bps budget. Confidence thresholds or position caps on LLM views would reduce churn. BL-Momentum at 0.212 is the leanest alternative.

### Interpretation

A returns-only LLM view generator feeding a disciplined Black-Litterman allocator delivered the best risk-adjusted performance of the BL family and showed positive, out-of-sample-concentrated forward-predictive skill with no detectable hindsight fingerprint. This validates the **method** — not yet a deployable edge — at a sample size too small to be conclusive.

The IC temporal profile (Fig B) runs opposite to a training-data leakage signal: LLM skill is weakest early in the backtest and strongest past the training cutoff. This is encouraging. The main practical constraints are turnover and the limitation of returns-only evidence.

### Next steps

1. **Rich-evidence layer (v3):** TIMESTAMPED archive (EODHD, Bloomberg) sliceable to ≤asof — live web search (Tavily) injects future into historical refits and is invalid for backtesting.
2. **Larger-N / multi-path:** Extend the OOS window or bootstrap to build IC inference beyond N ≈ 40 monthly periods.
3. **Turnover management:** Confidence thresholds and position caps to bring LLM-generator turnover within the 10 bps budget.
4. **Exact cutoff verification:** Obtain precise model training-data cutoff dates to sharpen the post-cutoff window.

## 8 — Lookahead Diagnostic (Hindsight-Tell)

A model exploiting training-data hindsight should predict **well early** (deep inside its training window) and **decay** toward the knowledge cutoff. Genuine point-in-time skill should be roughly stable or stronger post-cutoff. Momentum serves as the control: a deterministic, non-LLM signal with no cutoff-aligned decay structure.

**Method.** Spearman rank correlation (**IC**) between view vector **Q** and realised forward asset returns, computed per monthly refit. Restricted to assets with expressed views; replayed from cache — **no new API calls**.

**Fig B** is the key diagnostic: does LLM IC decline as refit dates enter the training-cutoff zone, while momentum IC does not?

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from scipy.stats import spearmanr

from aiam.llm.cache import PromptCache
from aiam.llm.views import LLMViewGenerator
from aiam.estimators.views import momentum_views
from aiam.data.split import TEST_START
from aiam.dl.walkforward import generate_refit_dates

# Load raw prices + compute daily returns (same source as the backtest scripts)
_prices_raw = pd.read_parquet(ROOT / "data/cache/prices_29.parquet")
_prices_raw = _prices_raw[_prices_raw.index.dayofweek < 5]
returns = _prices_raw.pct_change().dropna(how="all")

# Monthly refit grid (identical to run_bl_views_backtest.py)
_test_end   = returns.index[-1]
refit_dates = generate_refit_dates(
    TEST_START, _test_end, cadence="monthly", calendar=returns.index
)
print(f"Returns   : {returns.shape}  ({returns.index[0].date()} → {returns.index[-1].date()})")
print(f"Refit dates: {len(refit_dates)}  ({refit_dates[0].date()} → {refit_dates[-1].date()})")

In [ ]:
# CacheOnlyClient — raises on any live API call; ensures replay is cache-only
class CacheOnlyClient:
    def __init__(self, model: str) -> None:
        self.model = model

    def complete(self, prompt: str, *, system: str | None = None) -> str:
        raise RuntimeError(
            f"cache miss for model={self.model!r} — "
            "skipping this refit date (no API call permitted)"
        )

_cache = PromptCache(ROOT / "data/cache/llm")

_llm_gens = {
    "llm-anthropic": LLMViewGenerator(CacheOnlyClient("claude-opus-4-7"), cache=_cache),
    "llm-openai":    LLMViewGenerator(CacheOnlyClient("gpt-5.5"),          cache=_cache),
}

# per_refit_views: gen_name → [(refit_date, [asset_names], Q_array), ...]
per_refit_views: dict[str, list] = {
    g: [] for g in ["llm-anthropic", "llm-openai", "momentum"]
}
n_skipped = {"llm-anthropic": 0, "llm-openai": 0}

for _rd in refit_dates:
    hist = returns.loc[returns.index <= _rd]
    if len(hist) < 252:
        continue

    # Replay LLM generators (cache hits only; cache misses → empty Q → skipped)
    for gen_name, gen in _llm_gens.items():
        P, Q, _ = gen(hist, _rd)
        if len(Q) == 0:
            n_skipped[gen_name] += 1
            continue
        # Recover asset names from the single-1-per-row P matrix
        asset_idxs = [int(np.argmax(P[i])) for i in range(len(Q))]
        assets     = [returns.columns[j] for j in asset_idxs]
        per_refit_views[gen_name].append((_rd, assets, Q))

    # Momentum: deterministic, covers full 29-asset universe
    _, Q_mom, _ = momentum_views(hist, _rd)
    per_refit_views["momentum"].append((_rd, list(returns.columns), Q_mom))

for g, n in n_skipped.items():
    if n:
        print(f"WARNING: {g} — {n} refit date(s) had no cached response (skipped)")

for g, recs in per_refit_views.items():
    print(f"  {g:20s}: {len(recs)} refit dates with valid views")

In [ ]:
# Per-refit realized forward returns and Spearman IC computation
def _forward_returns(rd, next_rd, rets_df):
    mask = (rets_df.index > rd) & (rets_df.index <= next_rd)
    window = rets_df.loc[mask]
    return None if window.empty else (1 + window).prod() - 1

# Build refit → next-refit map
next_refit_map = {}
for _i, _rd in enumerate(refit_dates[:-1]):
    next_refit_map[_rd] = refit_dates[_i + 1]
# Last refit: use ~21 trading days forward (or data end)
_last_rd = refit_dates[-1]
_future = returns.index[returns.index > _last_rd]
if len(_future) >= 1:
    next_refit_map[_last_rd] = _future[min(20, len(_future) - 1)]

ic_records = []
for gen_name, records in per_refit_views.items():
    for (rd, assets, Q_views) in records:
        if rd not in next_refit_map or len(Q_views) < 2:
            continue
        fwd = _forward_returns(rd, next_refit_map[rd], returns[list(assets)])
        if fwd is None or fwd.isna().any():
            continue
        rho, _ = spearmanr(Q_views, fwd.values)
        if np.isfinite(rho):
            ic_records.append({"generator": gen_name, "refit_date": rd, "ic": rho})

ic_df = pd.DataFrame(ic_records)
if not ic_df.empty:
    ic_df["refit_date"] = pd.to_datetime(ic_df["refit_date"])

# Summary table
ic_summary = (
    ic_df.groupby("generator")["ic"]
    .agg(mean_ic="mean", ic_std="std", n_refits="count")
    .reindex(["llm-anthropic", "llm-openai", "momentum"])
)
ic_summary.columns = ["Mean IC", "IC Std", "N"]
print("Information Coefficient summary (Spearman rank, view Q vs. realised forward return)")
print(ic_summary.round(4))

In [ ]:
# Figure A — mean IC bar chart with ±1 SE error bars
if ic_df.empty:
    print("No IC data available — skipping Figure A.")
else:
    gen_order = ["llm-anthropic", "llm-openai", "momentum"]
    ic_agg = (
        ic_df.groupby("generator")["ic"]
        .agg(mean_ic="mean", ic_std="std", n="count")
        .reindex(gen_order)
    )
    ic_agg["se"] = ic_agg["ic_std"] / np.sqrt(ic_agg["n"])

    fig, ax = plt.subplots(figsize=(6, 3.8))
    x = np.arange(len(gen_order))
    bar_colors = [COLORS.get(g, "#888") for g in gen_order]
    labels_short = [DISPLAY.get(g, g) for g in gen_order]

    ax.bar(
        x,
        ic_agg["mean_ic"].fillna(0),
        yerr=ic_agg["se"].fillna(0),
        color=bar_colors, width=0.5, capsize=5, alpha=0.85,
        error_kw={"linewidth": 1.2},
    )
    ax.axhline(0, color="#555", linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(labels_short, fontsize=9)
    ax.set_ylabel("Mean Spearman IC")
    ax.set_title("Fig A — Mean IC by generator  (±1 SE,  N ≈ 40 monthly refits)", fontsize=10)
    ax.grid(True, axis="y", alpha=0.25)
    fig.tight_layout()
    plt.show()

In [ ]:
# Figure B — IC per refit chronologically with rolling mean and cutoff annotation
if ic_df.empty:
    print("No IC data available — skipping Figure B.")
else:
    fig, ax = plt.subplots(figsize=(16, 7.5))

    x_start = ic_df["refit_date"].min() - pd.DateOffset(months=1)
    x_end   = ic_df["refit_date"].max() + pd.DateOffset(months=2)
    ax.set_xlim(x_start, x_end)

    # Approximate training-cutoff zone (shaded band + vertical lines)
    cutoff_openai    = pd.Timestamp("2025-08-01")   # GPT-5.5 ≈ mid-2025
    cutoff_anthropic = pd.Timestamp("2026-01-01")   # claude-opus-4-7 ≈ 2026-01
    ax.axvspan(cutoff_openai, x_end, alpha=0.07, color="#aaaaaa",
               label="Approx. training-cutoff zone")
    ax.axvline(cutoff_openai,    color=COLORS["llm-openai"],    linestyle=":",
               linewidth=1.4, label="GPT-5.5 cutoff (≈ 2025-08, approx.)")
    ax.axvline(cutoff_anthropic, color=COLORS["llm-anthropic"], linestyle=":",
               linewidth=1.4, label="claude-opus-4-7 cutoff (≈ 2026-01, approx.)")

    colors_ic = {
        "llm-anthropic": COLORS["llm-anthropic"],
        "llm-openai":    COLORS["llm-openai"],
        "momentum":      COLORS["momentum"],
    }
    labels_ic = {
        "llm-anthropic": "BL-LLM(Anthropic) IC",
        "llm-openai":    "BL-LLM(OpenAI) IC",
        "momentum":      "Momentum IC (control)",
    }

    for gen in ["momentum", "llm-anthropic", "llm-openai"]:
        sub = ic_df[ic_df["generator"] == gen].sort_values("refit_date")
        if sub.empty:
            continue
        ax.scatter(sub["refit_date"], sub["ic"], s=25, alpha=0.50,
                   color=colors_ic[gen], zorder=3)
        # 4-period rolling mean overlay
        roll = sub.set_index("refit_date")["ic"].rolling(4, min_periods=2).mean()
        ax.plot(roll.index, roll.values, color=colors_ic[gen], linewidth=2.2,
                label=labels_ic[gen], zorder=4)

    ax.axhline(0.0, color="#777", linewidth=0.8, linestyle="--")
    ax.set_title(
        "Fig B — IC per refit (4-period rolling mean). LLM IC is low early (deepest\n"
        "in-training) and strongest in the post-cutoff window — opposite of a hindsight\n"
        "fingerprint. N small, IC noisy; suggestive, not dispositive.",
        fontsize=11,
    )
    ax.set_xlabel("Refit date", fontsize=12)
    ax.set_ylabel("Spearman IC  (view Q vs. realised forward return)", fontsize=12)
    ax.tick_params(axis="both", labelsize=11)
    ax.legend(fontsize=10, loc="upper left")
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax.tick_params(axis="x", rotation=45)
    ax.grid(True, alpha=0.2)
    fig.tight_layout()
    plt.show()

In [ ]:
# Leakage ratio: mean LLM IC / mean momentum IC
mean_ics = ic_df.groupby("generator")["ic"].mean()
mom_ic   = mean_ics.get("momentum", np.nan)

print("=" * 60)
print("Leakage ratio  (mean LLM IC / mean momentum IC)")
print("=" * 60)
for gen in ["llm-anthropic", "llm-openai"]:
    llm_ic = mean_ics.get(gen, np.nan)
    if np.isfinite(mom_ic) and abs(mom_ic) > 1e-6:
        ratio = llm_ic / mom_ic
        if ratio > 2.0:
            flag = "  ← >> 1 (elevated; Fig B shows no cutoff-aligned decay)"
        elif ratio > 1.2:
            flag = "  ← moderately elevated"
        elif ratio < 0.5:
            flag = "  ← below momentum baseline"
        else:
            flag = "  ← near parity"
    else:
        ratio = np.nan
        flag = "  (momentum IC ≈ 0; ratio undefined)"
    ratio_str = f"{ratio:+.3f}" if np.isfinite(ratio) else "  n/a"
    print(f"  {gen:20s}: {llm_ic:+.4f} / {mom_ic:+.4f} = {ratio_str}{flag}")
print()
print("Interpretation guide:")
print("  ratio >> 1 AND cutoff-aligned IC decay  → hindsight fingerprint")
print("  ratio >> 1 WITHOUT decay                → LLM skill genuine or noisy N")
print("  ratio ≈ 1  (with or without decay)      → inconclusive on this measure")
print("  ratio << 1                               → LLM views noisier than momentum")

### Interpretation

The IC numbers above are based on a single out-of-sample path (N ≈ 40 monthly periods). With this sample size, individual IC estimates are noisy; the following observations are **indicative, not dispositive**.

**IC level.** Both LLM providers exhibit positive mean IC (+0.117 Anthropic, +0.105 OpenAI) versus +0.042 for momentum. The LLM-vs-momentum difference is not statistically significant at this sample size, but the directional pattern holds across all 41 refit dates and both providers.

**Temporal decay (the key test).** A hindsight fingerprint would manifest as LLM IC being meaningfully positive early in the backtest (deep inside training data) and decaying toward zero or negative as refit dates approach the training-cutoff zone. **The observed pattern in Fig B runs opposite to this:** LLM IC is lowest in 2022–23 (deepest in the training window) and strongest in 2025–26 (at or past the training cutoffs). This is the anti-fingerprint pattern — skill concentrated out-of-sample — and softens the lookahead caveat substantially, though N ≈ 8 post-cutoff refit periods is too small for firm conclusions.

**Leakage ratio interpretation.** The ratios of +2.79× (Anthropic) and +2.50× (OpenAI) are elevated but are NOT combined with cutoff-aligned temporal decay. Per the interpretation guide, elevated ratio without decay is not the hindsight fingerprint. The alternative explanation — that LLM views carry genuine cross-sectional predictive content — is at least as consistent with the data.

**Limitations.** (1) N ≈ 41 refits provides low statistical power — IC standard errors are large. (2) Spearman IC measures per-refit cross-sectional rank prediction; it is related to but not the same as portfolio Sharpe contribution. (3) Training cutoff dates used as annotations are approximate; the actual data composition boundary may differ by weeks to months. A rigorous attribution would require a design with strictly held-out, unambiguously post-cutoff refit dates.